# Estudio de la complejidad a nivel instancia obtenida univariantemente

Uno de los problemas cuando se dispone de muchas variables es la redundancia entre ellas. En particular, con la correlación lineal de Pearson se pueden detectar aquellas variables que son linealmente dependientes y, por tanto, descartar las redundantes. Esto no es tan fácil de detectar en el caso no lineal.

Vamos a analizar la relación existente entre la complejidad a nivel instancia (obtenida de forma univariante) que otorgan las distintas variables y su relación con la correlación lineal. La idea es estudiar este comportamiento y ver si logramos algo para FS. En mi mente, variables relacionadas (ya sea lineal o no linealmente) deberían otorgar complejidades similares y podría servirnos para hacer un filtro inicial de variables. Todo esto es en mi cabeza, ahora lo vamos a estudiar.

Las funciones las cogemos del script UnivariateInstanceComplexityAnalysis_Code.py

In [1]:

import copy
from sklearn import preprocessing
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from skrebate import ReliefF
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import glob
import re
from UnivariateInstanceComplexityAnalysis_Code import *


In [2]:
import os
os.chdir("..")
root_path = os.getcwd()

In [3]:
# d1_uni = pd.read_csv("Results_UnivariateRanking_CM/ArtificialDataset1_featuresComplexityInstances.csv", index_col=None)
# d1_uni


In [4]:
dataset_files_all = ['Results_UnivariateRanking_CM/ArtificialDataset1_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset2_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset3_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset4_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset5_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset6_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset7_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset8_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset9_featuresComplexityInstances.csv',
                'Results_UnivariateRanking_CM/ArtificialDataset10_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset11_featuresComplexityInstances.csv',
                  'Results_UnivariateRanking_CM/ArtificialDataset12_featuresComplexityInstances.csv',
                'Results_UnivariateRanking_CM/ArtificialDataset13_featuresComplexityInstances.csv',
                'Results_UnivariateRanking_CM/ArtificialDataset14_featuresComplexityInstances.csv',
                    'Results_UnivariateRanking_CM/ArtificialDataset15_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset16_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset17_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset18_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset19_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset20_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset21_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset22_featuresComplexityInstances.csv']
                 #'Results_UnivariateRanking_CM/ArtificialDataset23_featuresComplexityInstances.csv'] # se queda muy pillado

In [6]:
summary_df = analyze_variable_relationships(dataset_files_all,show_plots=False)
summary_df

ArtificialDataset1
ArtificialDataset2
ArtificialDataset3
ArtificialDataset4
ArtificialDataset5
ArtificialDataset6
ArtificialDataset7
ArtificialDataset8
ArtificialDataset9
ArtificialDataset10
ArtificialDataset11
ArtificialDataset12
ArtificialDataset13
ArtificialDataset14
ArtificialDataset15
ArtificialDataset16
ArtificialDataset17
ArtificialDataset18
ArtificialDataset19
ArtificialDataset20
ArtificialDataset21
ArtificialDataset22


,dataset,measure,pearson_max,pearson_min,pearson_mean,pearson_median,pearson_std,spearman_max,spearman_min,spearman_mean,spearman_median,spearman_std,euclid_max,euclid_min,euclid_mean,euclid_median,euclid_std
0,ArtificialDataset1,Hostility,0.904561,-0.182604,0.013126,0.001449,0.117501,0.894644,-0.180470,0.010277,-0.004167,0.111071,11.069503,3.181336,7.607875,7.863358,1.740074
1,ArtificialDataset1,N1,0.366644,-0.084456,0.005099,0.001122,0.053631,0.360292,-0.092855,0.005161,0.001352,0.053369,17.606817,13.009612,16.293350,16.324828,0.622197
2,ArtificialDataset1,kDN,0.639411,-0.129386,0.004535,-0.002061,0.086482,0.633626,-0.129136,0.003982,-0.003567,0.084347,14.028542,6.283311,11.315613,11.088733,1.200451
3,ArtificialDataset2,Hostility,0.725754,-0.184165,0.013716,0.002047,0.108349,0.717344,-0.161800,0.009589,-0.003470,0.100830,9.730461,4.083331,6.885598,7.082812,1.290816
4,ArtificialDataset2,N1,0.323839,-0.147692,0.006456,0.003156,0.049209,0.323339,-0.148439,0.006622,0.003787,0.049032,17.306068,13.009612,16.095522,16.109866,0.478758
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,ArtificialDataset21,N1,0.205060,-0.062247,0.068845,0.068883,0.029642,0.204841,-0.062967,0.068701,0.068729,0.029658,16.904142,14.124447,15.535875,15.532225,0.297355
62,ArtificialDataset21,kDN,0.282665,0.022651,0.157803,0.157878,0.030331,0.284801,0.025516,0.157292,0.157357,0.030495,11.016351,8.544004,9.784949,9.783660,0.288579
63,ArtificialDataset22,Hostility,0.242637,-0.065339,0.079666,0.079613,0.029949,0.234681,-0.057235,0.086466,0.086433,0.030244,9.953493,5.642152,8.827170,8.836147,0.263440
64,ArtificialDataset22,N1,0.182516,-0.113239,0.041057,0.041055,0.030419,0.183338,-0.111911,0.041030,0.041023,0.030422,17.088007,14.239031,15.649818,15.652476,0.302554


Analizar comportamiento, en datasets mayores se aprecian menos diferencias. ESTUDIAR ESTO

In [ ]:
dataset_files = ['Results_UnivariateRanking_CM/ArtificialDataset1_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset2_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset3_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset4_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset5_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset6_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset7_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset8_featuresComplexityInstances.csv',
                 'Results_UnivariateRanking_CM/ArtificialDataset9_featuresComplexityInstances.csv',
                'Results_UnivariateRanking_CM/ArtificialDataset10_featuresComplexityInstances.csv']